# 04 · Headline evaluation on IxIDN-Matched

AUROC of the frozen 5-seed DisAtlas ensemble (Results; Supplementary *Seed and sampling uncertainty*, *Performance by rarity stratum*). Reads the frozen per-pair scores in `data/08S_eval/ixidn4364_scores.tsv`; 95% disease-cluster bootstrap intervals are read from the frozen metric tables (`ixidn4364_ci.tsv`), not resampled here.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
pd.set_option("display.width", 160)
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "00_raw").is_dir())
D = REPO / "data"
RAW = D / "00_raw"
def pairs(a, b):
    return {tuple(sorted(x)) for x in zip(a, b)}
from sklearn.metrics import roc_auc_score, average_precision_score
r3 = lambda x: float(f"{x:.3f}")

In [2]:
s = pd.read_csv(D / "08S_eval/ixidn4364_scores.tsv", sep="\t")
st = pd.read_csv(D / "08S_eval/ixidn4364_ci.tsv", sep="\t").set_index("stratum")
seeds = [c for c in s.columns if c.startswith("p_s")]
assert np.allclose(s[seeds].mean(axis=1), s.p)
test = s[s.pool == "test"]
a_all, a_test = roc_auc_score(s.y, s.p), roc_auc_score(test.y, test.p)
print(f"all {len(s)} pairs: AUROC {a_all:.3f} (CI {st.loc['all pairs', 'ci_lo']:.3f}-{st.loc['all pairs', 'ci_hi']:.3f}), AUPRC {average_precision_score(s.y, s.p):.3f}")
print(f"test split {len(test)} pairs: AUROC {a_test:.3f} (CI {st.loc['test split', 'ci_lo']:.3f}-{st.loc['test split', 'ci_hi']:.3f})")
assert (r3(a_all), r3(a_test)) == (0.765, 0.782)
assert np.isclose(st.loc["all pairs", "auroc"], a_all) and np.isclose(st.loc["test split", "auroc"], a_test)

all 4364 pairs: AUROC 0.765 (CI 0.729-0.799), AUPRC 0.777
test split 1094 pairs: AUROC 0.782 (CI 0.721-0.841)


## Seed variability
Supplementary: *SD of AUROC across five training seeds is 0.0015*.

In [3]:
per_seed = pd.Series({c[2:]: roc_auc_score(s.y, s[c]) for c in seeds})
print(per_seed.round(4).to_dict(), f"SD {per_seed.std():.4f}")
assert round(per_seed.std(), 4) == 0.0015

{'s4707': 0.7574, 's11': 0.7593, 's202': 0.7616, 's3407': 0.7589, 's8191': 0.7585} SD 0.0015


## Rarity strata
Supplementary table *Stratified AUROC by disease-rarity pairing*: rare–rare 2,452 pairs 0.722, rare–common 1,241 pairs 0.758, common–common 671 (645/26) 0.804; test split rare–rare 0.743 (301/321), rare–common 0.799 (68/279).

In [4]:
rows = []
for name, df in [("all", s), ("test", test)]:
    for k, g in df.groupby("rare_common_pair"):
        if (name, k) == ("test", "common-common"):
            continue  # 3 negatives only; not estimated in the manuscript
        rows.append((name, k, int(g.y.sum()), int((1 - g.y).sum()), r3(roc_auc_score(g.y, g.p)) if g.y.nunique() == 2 else None))
rc = pd.DataFrame(rows, columns=["pairs", "stratum", "n_pos", "n_neg", "auroc"])
ci = st.loc[["rare-rare", "rare-common", "common-common"], ["ci_lo", "ci_hi"]].round(3)
print(rc.to_string(index=False)); print(ci)
assert ci.loc['rare-rare'].tolist() == [0.672, 0.772] and ci.loc['rare-common'].tolist() == [0.668, 0.835] and ci.loc['common-common'].tolist() == [0.655, 0.933]
got = rc.set_index(["pairs", "stratum"])
assert got.loc[("all", "rare-rare"), "auroc"] == 0.722 and got.loc[("all", "rare-common"), "auroc"] == 0.758 and got.loc[("all", "common-common"), "auroc"] == 0.804
assert tuple(got.loc[("all", "common-common"), ["n_pos", "n_neg"]]) == (645, 26)
assert got.loc[("test", "rare-rare"), "auroc"] == 0.743 and got.loc[("test", "rare-common"), "auroc"] == 0.799
assert tuple(got.loc[("test", "rare-rare"), ["n_pos", "n_neg"]]) == (301, 321) and tuple(got.loc[("test", "rare-common"), ["n_pos", "n_neg"]]) == (68, 279)

pairs       stratum  n_pos  n_neg  auroc
  all common-common    645     26  0.804
  all   rare-common    253    988  0.758
  all     rare-rare   1284   1168  0.722
 test   rare-common     68    279  0.799
 test     rare-rare    301    321  0.743
               ci_lo  ci_hi
stratum                    
rare-rare      0.672  0.772
rare-common    0.668  0.835
common-common  0.655  0.933


## Other strata
Pair counts by split pool and by oncology label (either disease labelled cancer: 1,740 pairs; neither: 2,624). AUROCs for these subsets other than the held-out test split are not reported in the manuscript body and are not computed here.

In [5]:
for col in ["pool", "either_is_cancer"]:
    print(s.groupby(col).y.agg(n="size", n_pos="sum"))
assert s.either_is_cancer.value_counts().to_dict() == {0: 2624, 1: 1740}

          n  n_pos
pool              
cross  2167   1069
test   1094    491
tune   1103    622
                     n  n_pos
either_is_cancer             
0                 2624   1796
1                 1740    386


## Score separation

In [6]:
print(s.groupby("y").p.describe()[["count", "25%", "50%", "75%"]].round(3))

    count    25%    50%    75%
y                             
0  2182.0  0.002  0.007  0.046
1  2182.0  0.017  0.265  0.827
